# German Traffic Sign Recognition using a CNN with Optimizations

This notebook provides a complete workflow for training a Convolutional Neural Network (CNN) on the German Traffic Sign Recognition Benchmark (GTSRB). It incorporates several best practices and optimizations to ensure stable, efficient training and high accuracy.

### Step 1: Setup and GPU Configuration

In [ ]:
# Install required libraries
!pip install tensorflow opencv-python matplotlib seaborn scikit-learn pillow kaggle

# Import libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import os
import time

# OPTIMIZATION: GPU Configuration
print("GPU Configuration:")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"{len(gpus)} GPU(s) available and configured.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found, using CPU.")

print(f"TensorFlow version: {tf.__version__}")

### Step 2: Download and Prepare GTSRB Dataset
This dataset uses CSV files to map image paths to labels. We will download the data from Kaggle and then use pandas to manage the data loading process.

In [ ]:
# Setup Kaggle API for dataset download
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        print("Kaggle API configured successfully.")
    else:
        print("kaggle.json not found. Please upload the file.")
else:
    print("Kaggle API already configured.")

# Download and unzip the dataset
if not os.path.exists('/content/Train'):
    print("Downloading GTSRB dataset...")
    !kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign -p /content
    !unzip -q /content/gtsrb-german-traffic-sign.zip -d /content
    print("Dataset downloaded and unzipped.")
else:
    print("Dataset already exists.")

In [ ]:
# NEW: Data loading function for CSV-based datasets
def load_data_from_csv(csv_path, base_image_path, img_size=(48, 48)):
    """Loads images and labels from a CSV file."""
    data = pd.read_csv(csv_path)
    images = []
    labels = []

    print(f"Loading {len(data)} images from {csv_path}...")
    for index, row in data.iterrows():
        # Construct the full image path
        img_path = os.path.join(base_image_path, row['Path'])
        try:
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}, skipping...")
                continue
            # Resize and normalize
            img = cv2.resize(img, img_size)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype('float32') / 255.0

            images.append(img)
            labels.append(row['ClassId'])
        except Exception as e:
            print(f"Error loading {img_path}: {e}, skipping...")

    return np.array(images), np.array(labels)

# Define paths
base_dir = '/content/'
train_csv = os.path.join(base_dir, 'Train.csv')
test_csv = os.path.join(base_dir, 'Test.csv')
meta_csv = os.path.join(base_dir, 'Meta.csv')

# Load the data
start_time = time.time()
X_train_full, y_train_full = load_data_from_csv(train_csv, base_dir)
X_test, y_test = load_data_from_csv(test_csv, base_dir)
load_time = time.time() - start_time

# Get class information
num_classes = len(np.unique(y_train_full))
class_names = [f"Class {i}" for i in range(num_classes)]

print(f"\nData loading completed in {load_time:.2f} seconds")
print(f"Number of classes: {num_classes}")
print(f"Total training/validation images loaded: {len(X_train_full)}")
print(f"Total test images loaded: {len(X_test)}")
print(f"Image shape: {X_train_full.shape[1:]}")

### Step 3: Create Validation Set and Configure Augmentation
The dataset provides a training and a test set. We need to create our own validation set by splitting the training data.

In [ ]:
# Split the full training set into training (80%) and validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, # 20% for validation
    random_state=42,
    stratify=y_train_full # Ensure proportional class representation
)

print(f"Dataset Split:")
print(f"Training set:   {len(X_train)} images")
print(f"Validation set: {len(X_val)} images")
print(f"Test set:       {len(X_test)} images")

# Data augmentation for better generalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False, # Flipping traffic signs is not a good idea
    fill_mode='nearest'
)

print("\nData augmentation configured.")

### Step 3.5: CRITICAL - Visually Verify Data and Labels
Let's check if the images and labels loaded correctly. If the labels don't match the images, something is wrong with the data pipeline.

In [ ]:
# Plot a few random images from the training set with their labels
plt.figure(figsize=(16, 10))
for i in range(12):
    ax = plt.subplot(3, 4, i + 1)
    idx = np.random.randint(0, len(X_train))
    plt.imshow(X_train[idx])
    correct_label = class_names[y_train[idx]]
    plt.title(f"Label: {correct_label}", fontsize=12)
    plt.axis("off")

plt.tight_layout()
plt.show()

### Step 4: Build the CNN Model
We'll use a robust architecture with Batch Normalization for stability and L2 Regularization to prevent overfitting.

In [ ]:
# Robust CNN with L2 regularization and more dropout
from tensorflow.keras import models # Import models here

def create_robust_cnn(input_shape, num_classes):
    model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # 🔽 Replace Flatten + Dense(512) with GAP
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    # Output layer
    layers.Dense(85, activation='softmax')
])
    return model

model = create_robust_cnn((48, 48, 3), num_classes)

# Compile with a stable learning rate and gradient clipping
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Architecture:")
model.summary()

### Step 5: Training Setup

In [ ]:
# Callbacks for smart training
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10, # Stop after 10 epochs of no improvement
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5, # Reduce LR after 5 epochs of no improvement
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'gtsrb_best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, reduce_lr, checkpoint]

BATCH_SIZE = 64
EPOCHS = 50 # Early stopping will likely finish before this

print(f"Training Configuration:")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")

### Step 6: Train the Model

In [ ]:
print("Starting training...")
start_time = time.time()

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time/60:.2f} minutes.")

In [ ]:
# Plot training and validation history
def plot_training_history(history):
    pd.DataFrame(history.history).plot(figsize=(10, 6))
    plt.grid(True)
    plt.gca().set_ylim(0, 1) # Set accuracy y-axis limit
    plt.title('Model Training History')
    plt.xlabel('Epoch')
    plt.show()

plot_training_history(history)

### Step 7: Model Evaluation on the Test Set

In [ ]:
# Load the best model saved by ModelCheckpoint
print("Loading best model for final evaluation...")
model = keras.models.load_model('gtsrb_best_model.keras')

# Evaluate on the unseen test set
print("\nEvaluating on test data...")
loss, accuracy = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=0)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Test Loss: {loss:.4f}")

# Generate classification report
y_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

### Step 8: Visualize Performance

In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(18, 18))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.show()

In [ ]:
# Show some sample predictions with their confidence
def show_predictions(X, y_true, y_pred_proba, class_names, num_samples=16):
    y_pred = np.argmax(y_pred_proba, axis=1)
    rows = (num_samples + 3) // 4
    plt.figure(figsize=(16, 4 * rows))

    indices = np.random.choice(len(X), num_samples, replace=False)

    for i, idx in enumerate(indices):
        plt.subplot(rows, 4, i + 1)
        plt.imshow(X[idx])
        true_label = class_names[y_true[idx]]
        pred_label = class_names[y_pred[idx]]
        confidence = np.max(y_pred_proba[idx]) * 100

        color = 'green' if true_label == pred_label else 'red'
        title = f"True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%"
        plt.title(title, color=color)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

show_predictions(X_test, y_test, y_pred_proba, class_names)

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image

def predict_uploaded_image(model, img_size, class_names):
    """Uploads an image, preprocesses it, and makes a prediction."""
    uploaded = files.upload()

    for filename in uploaded.keys():
        try:
            # Load the image
            img = image.load_img(filename, target_size=img_size)
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) # Create a batch
            img_array /= 255.0 # Normalize

            # Make prediction
            predictions = model.predict(img_array)
            predicted_class_id = np.argmax(predictions[0])
            confidence = np.max(predictions[0]) * 100
            predicted_class_name = class_names[predicted_class_id]

            print(f"\nUploaded image: {filename}")
            print(f"Predicted class: {predicted_class_name} (Class ID: {predicted_class_id})")
            print(f"Confidence: {confidence:.2f}%")

            # Optional: Display the image
            plt.imshow(img)
            plt.title(f"Prediction: {predicted_class_name} ({confidence:.2f}%)")
            plt.axis('off')
            plt.show()

        except Exception as e:
            print(f"Error processing image {filename}: {e}")

# Define image size (should match the model's input size)
img_size = (48, 48) # Matches the input shape defined in create_robust_cnn

# Predict on the uploaded image
predict_uploaded_image(model, img_size, class_names)

In [ ]:
# Download and unzip the new dataset
dataset_name = "harbhajansingh21/german-traffic-sign-dataset"
new_dataset_dir = '/content/new_gtsrb/'

if not os.path.exists(new_dataset_dir):
    print(f"Downloading {dataset_name} dataset...")
    !kaggle datasets download -d {dataset_name} -p /content/
    !mkdir -p {new_dataset_dir}
    !unzip -q /content/german-traffic-sign-dataset.zip -d {new_dataset_dir}
    print("New dataset downloaded and unzipped.")
else:
    print("New dataset already exists.")

# List files in the new dataset directory to confirm
print("\nFiles in the new dataset directory:")
print(os.listdir(new_dataset_dir))

In [ ]:
# Function to load images from a directory structure
def load_images_from_directory(base_dir, img_size=(48, 48)):
    images = []
    labels = []
    # Assuming class folders are numerical (0, 1, 2, ...)
    class_dirs = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.isdigit()])

    print(f"Loading images from {base_dir}...")
    for class_id_str in class_dirs:
        class_id = int(class_id_str)
        class_dir_path = os.path.join(base_dir, class_id_str)
        for img_name in os.listdir(class_dir_path):
            img_path = os.path.join(class_dir_path, img_name)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Could not read image {img_path}, skipping...")
                    continue
                # Resize and normalize
                img = cv2.resize(img, img_size)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = img.astype('float32') / 255.0

                images.append(img)
                labels.append(class_id)
            except Exception as e:
                print(f"Error loading {img_path}: {e}, skipping...")

    return np.array(images), np.array(labels)

# Define the base directories for training and testing images
# Note: We will use the already loaded X_test and y_test for evaluation
train_images_dir = '/content/Train'

# Load images from the training directory if needed for further analysis (optional)
# start_time = time.time()
# X_dir_train, y_dir_train = load_images_from_directory(train_images_dir)
# load_time_dir_train = time.time() - start_time
# print(f"\nData loading from training directory completed in {load_time_dir_train:.2f} seconds")
# print(f"Total training images loaded from directories: {len(X_dir_train)}")

# Evaluate the current model on the ALREADY loaded test images (from Test.csv)
print("\nEvaluating on test data loaded from Test.csv...")
# Use the variables X_test and y_test which were loaded earlier
if 'X_test' in globals() and 'y_test' in globals():
    start_time = time.time()
    loss_dir, accuracy_dir = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=0)
    evaluation_time = time.time() - start_time
    print(f"\nEvaluation on test data completed in {evaluation_time:.2f} seconds")
    print(f"Test Accuracy (from Test.csv): {accuracy_dir:.4f} ({accuracy_dir*100:.2f}%)")
    print(f"Test Loss (from Test.csv): {loss_dir:.4f}")

    # Generate classification report for directory loaded data
    print("\nGenerating Classification Report on Test Data (from Test.csv):")
    start_time = time.time()
    y_dir_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_dir_pred = np.argmax(y_dir_pred_proba, axis=1)
    prediction_time = time.time() - start_time
    print(f"Prediction on test data completed in {prediction_time:.2f} seconds")


    print("\nClassification Report on Test Data (from Test.csv):")
    print(classification_report(y_test, y_dir_pred, target_names=class_names))
else:
    print("Error: X_test and y_test not found. Please ensure the cell loading data from Test.csv was executed.")